# Project 2 — Environment and Cloud setup

Run this notebook on the Mac's Jupyter kernel for local preflight and Cloud setup. Spark analysis and model training run only on the Dataproc Jupyter kernel.

The run is pinned to project `sg-rainfall-dsa5208-261011`, region `asia-southeast1`, Education billing account `01DAD8-AE5F1A-E3910A`, and the user-authorised credit ID `3LU63BPGY77JFDQQ`. Google applies credits automatically at billing-account scope; `gcloud` cannot attach a credit ID to an individual job. This workflow checks project/account linkage and records the authorised credit identity. It does not require Reports or Cost-table review.

In [ ]:
from pathlib import Path
import json
import os
import re
import subprocess

from IPython.display import display

PROJECT_ID = "sg-rainfall-dsa5208-261011"
BILLING_ACCOUNT_NAME = "Billing Account for Education"
BILLING_ACCOUNT_ID = "01DAD8-AE5F1A-E3910A"
REGION = "asia-southeast1"
CLUSTER = "dsa5208-rainfall"
BUCKET = "sg-rainfall-dsa5208-261011-data"
RUN_ID = "20261012-full-02"


In [ ]:
DATA_ROOT_VALUE = os.environ.get("DATA_ROOT")
if not DATA_ROOT_VALUE:
    raise RuntimeError("Set DATA_ROOT to the directory containing the raw/ input folder.")
DATA_ROOT = Path(DATA_ROOT_VALUE).expanduser()
if not (DATA_ROOT / "raw").is_dir():
    raise FileNotFoundError(DATA_ROOT / "raw")


def find_repo_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "scripts").is_dir() and (candidate / "manifests").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the repository root from this notebook.")


REPO_ROOT = find_repo_root()
print(f"Notebook root: {REPO_ROOT}")
print(f"Project: {PROJECT_ID}")
print(f"Billing account: {BILLING_ACCOUNT_NAME} ({BILLING_ACCOUNT_ID})")
print(f"Run ID: {RUN_ID}")


In [ ]:
REBUILD_MANIFEST = False  # Set True to rescan the eight local annual inputs.
manifest_path = REPO_ROOT / "manifests" / "input_manifest.json"
if REBUILD_MANIFEST:
    import importlib.util

    module_path = REPO_ROOT / "scripts" / "build_manifest.py"
    spec = importlib.util.spec_from_file_location("build_manifest", module_path)
    manifest_module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(manifest_module)
    rebuilt = manifest_module.build_manifest(DATA_ROOT)
    temporary_path = manifest_path.with_suffix(".json.tmp")
    temporary_path.write_text(json.dumps(rebuilt, indent=2) + "\n", encoding="utf-8")
    temporary_path.replace(manifest_path)
    print(f"Rebuilt manifest: {manifest_path}")


In [ ]:
if not manifest_path.is_file():
    raise FileNotFoundError(manifest_path)
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
manifest_rows = [
    {
        "year": item["year"],
        "rows": item["row_count"],
        "size_bytes": item["size_bytes"],
        "sha256": item["sha256"],
        "schema_matches": item["schema_matches_expected"],
        "malformed_rows": item["rows_with_wrong_field_count"],
        "invalid_timestamps": item["unparseable_or_missing_timestamps"],
    }
    for item in manifest["files"]
]
display(manifest_rows)
print(f"Input years: {manifest['selection']['years']}")
print(f"Files in manifest: {len(manifest_rows)}")
print("Raw CSV files are read-only.")


In [ ]:
CREATE_PROJECT = True

if CREATE_PROJECT:
    result = subprocess.run(
        ["gcloud", "projects", "describe", PROJECT_ID, "--format=json"],
        capture_output=True, text=True,
    )
    if result.returncode:
        created = subprocess.run(
            ["gcloud", "projects", "create", PROJECT_ID, "--name=DSA5208 Rainfall"],
            capture_output=True, text=True,
        )
        if created.returncode:
            raise RuntimeError("Project creation failed. Check project-creation permissions in Console.")
        print(f"Created project: {PROJECT_ID}")
    else:
        print(f"Project already exists: {PROJECT_ID}")

In [ ]:
import re

def run_gcloud(args, *, json_output=False):
    result = subprocess.run(
        ["gcloud", *args, *([f"--billing-project={PROJECT_ID}"] if args[:2] == ["billing", "budgets"] else [])],
        check=False,
        capture_output=True,
        text=True,
    )
    if result.returncode:
        message = re.sub(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+", "[redacted identity]", result.stderr.strip())
        raise RuntimeError(message or f"gcloud exited with {result.returncode}")
    if json_output:
        return json.loads(result.stdout or "null")
    if result.stdout.strip():
        print(result.stdout.strip())
    return result.stdout


In [ ]:
def billing_project_state():
    return run_gcloud(
        ["billing", "projects", "describe", PROJECT_ID, "--format=json"],
        json_output=True,
    )

before = billing_project_state()
target = f"billingAccounts/{BILLING_ACCOUNT_ID}"
print({
    "project_id": PROJECT_ID,
    "billing_enabled_before": before.get("billingEnabled"),
    "already_linked_to_education": before.get("billingAccountName") == target,
})


In [ ]:
LINK_PROJECT_TO_EDUCATION = True

def ensure_education_billing_link():
    current = billing_project_state()
    if current.get("billingAccountName") == target and current.get("billingEnabled") is True:
        return current, True

    try:
        run_gcloud([
            "billing", "projects", "link", PROJECT_ID,
            f"--billing-account={BILLING_ACCOUNT_ID}",
        ])
        verified = billing_project_state()
        billing_ready = (
            verified.get("billingAccountName") == target
            and verified.get("billingEnabled") is True
        )
    except RuntimeError as exc:
        message = str(exc)
        if "IAM_PERMISSION_DENIED" in message or "does not have permission" in message:
            print(
                "Billing link blocked: the active gcloud identity lacks "
                "project billing-assignment permissions. No bucket, cluster, or job was created."
            )
            return current, False
        raise
    return verified, billing_ready


In [ ]:
if LINK_PROJECT_TO_EDUCATION:
    verified, billing_ready = ensure_education_billing_link()
    display({
        "project_id": PROJECT_ID,
        "billing_account": BILLING_ACCOUNT_NAME,
        "billing_account_id": BILLING_ACCOUNT_ID,
        "billing_enabled": verified.get("billingEnabled"),
        "linkage_verified": verified.get("billingAccountName") == target,
    })
else:
    billing_ready = False
    print("Skipped billing linkage. No cloud resources may be created.")


In [ ]:
# Stop before provisioning if the project is not linked to the selected account.
if not billing_ready:
    raise RuntimeError(
        "Project billing is not enabled on the selected Education account. "
        "No cloud resources will be created."
    )
print("The selected project billing account is verified for this setup session.")


In [ ]:
ENABLE_APIS = True
REQUIRED_APIS = [
    "cloudresourcemanager.googleapis.com",
    "serviceusage.googleapis.com",
    "iap.googleapis.com",
    "dataproc.googleapis.com",
    "compute.googleapis.com",
    "storage.googleapis.com",
    "iam.googleapis.com",
    "cloudbilling.googleapis.com",
]

if ENABLE_APIS:
    for api in REQUIRED_APIS:
        run_gcloud(["services", "enable", api, f"--project={PROJECT_ID}"])
else:
    print("API enablement skipped.")


In [ ]:
# Read-only regional quota and the fixed Dataproc topology.
region_info = run_gcloud(
    ["compute", "regions", "describe", REGION, f"--project={PROJECT_ID}", "--format=json"],
    json_output=True,
)
display([
    quota for quota in region_info.get("quotas", [])
    if quota["metric"] in ("CPUS", "E2_CPUS", "DISKS_TOTAL_GB", "SSD_TOTAL_GB", "IN_USE_ADDRESSES")
])
display({
    "region": REGION,
    "image": "2.2.87-debian12",
    "master": {"machine_type": "e2-standard-4", "boot_disk_gb": 100},
    "workers": {"count": 2, "machine_type": "e2-standard-4", "boot_disk_gb_each": 75},
    "spark": {
        "event_log_enabled": True,
        "event_log_dir": f"gs://{BUCKET}/spark-history/",
    },
    "lifecycle": {"delete_when_idle": "45m", "maximum_age": "6h"},
})


In [ ]:
display({
    "project_id": PROJECT_ID,
    "billing_account": BILLING_ACCOUNT_NAME,
    "billing_account_id": BILLING_ACCOUNT_ID,
    "billing_linkage_verified": billing_ready,
})
print("Notebook setup uses this single selected account; science notebooks do not repeat the check.")


In [ ]:
NETWORK = "rainfall-private"
SUBNET_NAME = "rainfall-singapore"
VM_SERVICE_ACCOUNT = f"rainfall-vm@{PROJECT_ID}.iam.gserviceaccount.com"

def resource_exists(args):
    return subprocess.run(["gcloud", *args, "--format=json"], capture_output=True, text=True).returncode == 0

if not resource_exists(["compute", "networks", "describe", NETWORK, f"--project={PROJECT_ID}"]):
    run_gcloud(["compute", "networks", "create", NETWORK, "--subnet-mode=custom", f"--project={PROJECT_ID}"])
if not resource_exists(["compute", "networks", "subnets", "describe", SUBNET_NAME, f"--region={REGION}", f"--project={PROJECT_ID}"]):
    run_gcloud(["compute", "networks", "subnets", "create", SUBNET_NAME, f"--network={NETWORK}", f"--region={REGION}", "--range=10.20.0.0/24", "--enable-private-ip-google-access", f"--project={PROJECT_ID}"])
if not resource_exists(["compute", "firewall-rules", "describe", "rainfall-internal", f"--project={PROJECT_ID}"]):
    run_gcloud(["compute", "firewall-rules", "create", "rainfall-internal", f"--network={NETWORK}", "--source-ranges=10.20.0.0/24", "--allow=tcp,udp,icmp", f"--project={PROJECT_ID}"])
if not resource_exists(["iam", "service-accounts", "describe", VM_SERVICE_ACCOUNT, f"--project={PROJECT_ID}"]):
    run_gcloud(["iam", "service-accounts", "create", "rainfall-vm", "--display-name=Rainfall Dataproc VM", f"--project={PROJECT_ID}"])
# Dedicated VM identity; no service-account keys are created.
run_gcloud(["projects", "add-iam-policy-binding", PROJECT_ID, f"--member=serviceAccount:{VM_SERVICE_ACCOUNT}", "--role=roles/dataproc.worker", "--condition=None", "--format=none"])
print("Private subnet, internal-only firewall and dedicated VM identity prepared.")
if not resource_exists(["compute", "firewall-rules", "describe", "rainfall-iap-ssh", f"--project={PROJECT_ID}"]):
    run_gcloud(["compute", "firewall-rules", "create", "rainfall-iap-ssh", f"--network={NETWORK}", "--source-ranges=35.235.240.0/20", "--allow=tcp:22", "--target-tags=rainfall-cluster", f"--project={PROJECT_ID}"])
# Reuse the signed-in project owner for IAP; do not expose the human identity.
active_identity = subprocess.run(["gcloud", "config", "get-value", "account"], capture_output=True, text=True, check=True).stdout.strip()
run_gcloud(["projects", "add-iam-policy-binding", PROJECT_ID, f"--member=user:{active_identity}", "--role=roles/iap.tunnelResourceAccessor", "--condition=None", "--format=none"])
print("IAP SSH access prepared for the existing project owner; no public VM address.")


In [ ]:
CREATE_BUCKET = True
UPLOAD_INPUTS = True

if CREATE_BUCKET and not resource_exists(["storage", "buckets", "describe", f"gs://{BUCKET}"]):
    run_gcloud([
        "storage", "buckets", "create", f"gs://{BUCKET}",
        f"--project={PROJECT_ID}",
        f"--location={REGION}",
        "--default-storage-class=STANDARD",
        "--uniform-bucket-level-access",
    ])
    print(f"Created bucket gs://{BUCKET}")

run_gcloud([
    "storage", "buckets", "add-iam-policy-binding", f"gs://{BUCKET}",
    f"--member=serviceAccount:{VM_SERVICE_ACCOUNT}",
    "--role=roles/storage.objectAdmin", "--format=none",
])


In [ ]:
if UPLOAD_INPUTS:
    manifest_path = REPO_ROOT / "manifests" / "input_manifest.json"
    if not manifest_path.is_file():
        raise FileNotFoundError(manifest_path)
    manifest_uri = f"gs://{BUCKET}/runs/{RUN_ID}/config/input_manifest.json"
    if resource_exists(["storage", "objects", "describe", manifest_uri]):
        raise FileExistsError(f"Refusing to replace run manifest: {manifest_uri}")
    run_gcloud(["storage", "cp", str(manifest_path), manifest_uri])


In [ ]:
if UPLOAD_INPUTS:
    for record in manifest["files"]:
        year = record["year"]
        target_uri = f"gs://{BUCKET}/{record['relative_path']}"
        if resource_exists(["storage", "objects", "describe", target_uri]):
            object_info = run_gcloud(
                ["storage", "objects", "describe", target_uri, "--format=json"],
                json_output=True,
            )
            metadata = object_info.get("custom_fields", object_info.get("metadata", {}))
            if metadata.get("sha256") != record["sha256"]:
                raise RuntimeError(
                    f"Existing input has no matching SHA-256 metadata: {target_uri}"
                )
            print(f"Verified existing input: {target_uri}")
        else:
            source = DATA_ROOT / "raw" / f"HistoricalRainfallacrossSingapore{year}.csv"
            if not source.is_file():
                raise FileNotFoundError(source)
            run_gcloud([
                "storage", "cp", str(source), target_uri,
                f"--custom-metadata=sha256={record['sha256']}",
            ])
            print(f"Uploaded {year}: {record['size_bytes']} bytes")


In [ ]:
if UPLOAD_INPUTS:
    config_path = REPO_ROOT / "config" / "experiment.json"
    config_uri = f"gs://{BUCKET}/runs/{RUN_ID}/config/experiment.json"
    run_gcloud(["storage", "cp", str(config_path), config_uri])
    print(f"Inputs and experiment configuration uploaded for {RUN_ID}.")
else:
    print("Input upload skipped by configuration.")


In [ ]:
# Copy only the notebooks used for analysis into the Dataproc Jupyter bucket.
analysis_notebooks = [
    "01_clean_and_task1.ipynb",
    "02_build_features.ipynb",
    "03_train_evaluate.ipynb",
]
for name in analysis_notebooks:
    notebook_path = REPO_ROOT / "notebooks" / name
    run_gcloud([
        "storage", "cp", str(notebook_path),
        f"gs://{BUCKET}/notebooks/{name}",
    ])
print("Uploaded the three analysis notebooks. No source bundle or runtime ledger is required.")


In [ ]:
CREATE_CLUSTER = True
DATAPROC_IMAGE = "2.2.87-debian12"
SUBNET = SUBNET_NAME

if CREATE_CLUSTER and not resource_exists(["dataproc", "clusters", "describe", CLUSTER, f"--project={PROJECT_ID}", f"--region={REGION}"]):
    if not BUCKET or not SUBNET or not VM_SERVICE_ACCOUNT or not DATAPROC_IMAGE:
        raise RuntimeError("Set bucket, subnet, service account, and exact image first.")
    run_gcloud([
        "dataproc", "clusters", "create", CLUSTER,
        f"--project={PROJECT_ID}", f"--region={REGION}",
        f"--image-version={DATAPROC_IMAGE}",
        f"--subnet={SUBNET}", "--no-address",
        f"--service-account={VM_SERVICE_ACCOUNT}", "--scopes=cloud-platform",
        "--master-machine-type=e2-standard-4",
        "--master-boot-disk-type=pd-balanced", "--master-boot-disk-size=100GB",
        "--num-workers=2", "--worker-machine-type=e2-standard-4",
        "--worker-boot-disk-type=pd-balanced", "--worker-boot-disk-size=75GB",
        f"--bucket={BUCKET}", f"--temp-bucket={BUCKET}",
        "--tags=rainfall-cluster", "--enable-component-gateway", "--optional-components=JUPYTER",
        f"--properties=dataproc:jupyter.notebook.gcs.dir=gs://{BUCKET}/notebooks/,spark:spark.eventLog.enabled=true,spark:spark.eventLog.dir=gs://{BUCKET}/spark-history/",
        "--delete-max-idle=45m", "--delete-max-age=6h",
    ])
    print(f"Dataproc cluster {CLUSTER} is ready in {REGION}.")
else:
    print("Cluster already exists or creation was skipped; inspect its state before execution.")

In [ ]:
cluster_info = run_gcloud(["dataproc", "clusters", "describe", CLUSTER, f"--project={PROJECT_ID}", f"--region={REGION}", "--format=json"], json_output=True)
display({"cluster": CLUSTER, "state": cluster_info["status"]["state"], "master": cluster_info["config"]["masterConfig"]["instanceNames"], "component_urls": cluster_info["config"].get("endpointConfig", {}).get("httpPorts", {}), "auto_delete": cluster_info["config"].get("lifecycleConfig")})


In [ ]:
DELETE_CLUSTER = False

if DELETE_CLUSTER:
    run_gcloud([
        "dataproc", "clusters", "delete", CLUSTER,
        f"--project={PROJECT_ID}", f"--region={REGION}", "--quiet",
    ])
    remaining = run_gcloud([
        "dataproc", "clusters", "list",
        f"--project={PROJECT_ID}", f"--region={REGION}",
        "--format=json",
    ], json_output=True)
    display({"remaining_cluster_names": [item["clusterName"] for item in remaining]})
else:
    print("Cluster deletion skipped. Set DELETE_CLUSTER=True after outputs and notebooks are persisted.")

## Analysis notebooks

Run `01_clean_and_task1.ipynb`, `02_build_features.ipynb`, and `03_train_evaluate.ipynb` on the Dataproc Jupyter kernel. Each notebook records the Spark SQL settings it actually uses and writes its outputs under the selected run prefix.
